# DPPUv2 Interactive Visualizer  (v4ec — EC-Weyl Engine)

**Purpose:** Interactive visualization of DPPUv2 effective potentials and phase diagrams across topologies and parameter spaces, with EC-Weyl ($\alpha \cdot C^2_{\rm EC}$) coupling.

## What you can explore
| Control | Description |
|---|---|
| **Topology** | S³×S¹ · T³×S¹ · Nil³×S¹ |
| **Torsion Mode** | MX (mixed) · VT (vector-trace) · AX (axial) |
| **NY Variant** | FULL · TT-only · Ree-only |
| **θ_NY** | Nieh-Yan coupling constant |
| **ε (squashing)** | Geometric deformation of S³ or Nil³ |
| **α_W (Weyl)** | Coupling of EC-Weyl term $\alpha \cdot C^2_{\rm EC}$ in Lagrangian |

## EC-Weyl coupling α_W

The Lagrangian includes the EC-Weyl term:
$$L = \frac{R_{\rm EC}}{2\kappa^2} + \theta_{\rm NY} \cdot N + \alpha_W \cdot C^2_{\rm EC}$$

Key physical facts (EC version — differs from LC paper03 for Nil³):

| Topology | $C^2_{\rm LC}$ at $\varepsilon=0$ | $\alpha_W$ status |
|---|---|---|
| **T³×S¹** | $\equiv 0$ (flat torus) | **no effect**, regardless of $\varepsilon$ |
| **S³×S¹**, $\varepsilon=0$ | $= 0$ (conformally flat) | **inactive** |
| **S³×S¹**, $\varepsilon\neq 0$ | $\neq 0$ (squashed) | **active** |
| **Nil³×S¹**, any $\varepsilon$ | $= 4/(3R^4) \neq 0$ (non-conf.\ flat) | **always active** — EC paper main result |

> **EC vs LC difference:** In the LC paper03 viewer, Nil³ at $\varepsilon=0$ was listed as conformally flat ($C^2=0$, $\alpha$ inactive).  In EC theory, the background $C^2_{\rm LC}(Nil^3) = 4/(3R^4) \neq 0$ activates $\alpha_W$ even at $\varepsilon=0$, generating the EC slice minimum (Theorem 6).  The **Weyl status indicator** reflects this.

## Engine notes (EC)

- Uses `UnifiedEngine` (EC dppu package, `dppu.topology.unified`)
- $\varepsilon$ and $\alpha_W$ kept as SymPy symbols — one engine run per (topology, mode, NY variant)
- T³: $\alpha$ not a free symbol (flat); ignored silently
- Nil³: radial symbol is `R` (Nil³ scale), not `r`

**Requirements:** Python 3.10+, ipywidgets, NumPy, Matplotlib, SymPy, SciPy, dppu package

## Point Diagnosis (added 2026-04-04)

A **Point Diagnosis** panel is available below the potential plot.
It distinguishes two senses of equilibrium at any point $(r; V, \eta)$:

| Term | Definition | Paper sense |
|---|---|---|
| **Slice extremum** | $\partial_r V_{\rm eff} = 0$ at fixed $(\eta, V)$ | paper01 phase diagram |
| **Full stationary point** | $\nabla_{(r,\eta,V)} V_{\rm eff} = 0$ simultaneously | paper03 vacuum dictionary |

### Usage
1. Select an **Active Point** (dropdown in *Phase Diagram Controls*).
2. Click anywhere on the **potential plot** — the clicked $r$ is set instantly and diagnosis runs automatically.
   Or enter $r$ manually in the box and click **Diagnose This Point**.
3. Use **Find Nearest Extremum** to snap $r$ to the nearest slice minimum.

> **Note:** Diagnosis applies to the **Active Point only** (Points 2 and 3 are not diagnosed).

Tolerance is scale-adaptive: $\text{tol} = 10^{-3} \times \max(|V_{\rm eff}|, 1)$.


In [ ]:
# =============================================================================
# DPPUv2 Interactive Phase Diagram & Potential Viewer  (v4ec EC-Weyl Engine)
# =============================================================================
%matplotlib inline
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# ── Path setup ──────────────────────────────────────────────────────────────
# Add _DPPUv2_Phase2/ to sys.path so that `import dppu` works.
# This notebook is at:   scripts/visualize/
# dppu package is at:    ../../dppu/   (two levels up)
_phase2_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _phase2_root not in sys.path:
    sys.path.insert(0, _phase2_root)

# Also add the viewer's own directory (for the import below)
_viewer_dir = os.getcwd()
if _viewer_dir not in sys.path:
    sys.path.insert(0, _viewer_dir)

print(f"dppu root: {_phase2_root}")

# ── Import viewer ────────────────────────────────────────────────────────────
from DPPUv2_interactive_viewer_v4ec import DPPUv2InteractiveViewer, clear_cache

# =============================================================================
# Optional: Customise slider defaults BEFORE creating the viewer
# =============================================================================
# ── Phase Diagram Range ──
# DPPUv2InteractiveViewer.SLIDER_V_MAX_MIN     = 1.0     # default: 1.0
# DPPUv2InteractiveViewer.SLIDER_V_MAX_MAX     = 50.0    # default: 50.0
# DPPUv2InteractiveViewer.SLIDER_V_MAX_DEFAULT = 10.0    # default: 10.0

# DPPUv2InteractiveViewer.SLIDER_ETA_MIN_MIN     = -50.0  # default: -50.0
# DPPUv2InteractiveViewer.SLIDER_ETA_MIN_MAX     = 0.0    # default: 0.0
# DPPUv2InteractiveViewer.SLIDER_ETA_MIN_DEFAULT = -15.0  # default: -15.0

# DPPUv2InteractiveViewer.SLIDER_ETA_MAX_MIN     = 0.0    # default: 0.0
# DPPUv2InteractiveViewer.SLIDER_ETA_MAX_MAX     = 50.0   # default: 50.0
# DPPUv2InteractiveViewer.SLIDER_ETA_MAX_DEFAULT = 5.0    # default: 5.0

# ── Potential Plot Range (log-scale exponents) ──
# DPPUv2InteractiveViewer.SLIDER_R_MAX_MIN     = -2      # 10^-2, default: -2
# DPPUv2InteractiveViewer.SLIDER_R_MAX_MAX     = 6       # 10^6,  default: 6
# DPPUv2InteractiveViewer.SLIDER_R_MAX_DEFAULT = 5.0    # default: 5.0

# DPPUv2InteractiveViewer.SLIDER_VEFF_MIN_MIN     = 0    # 10^0,  default: 0
# DPPUv2InteractiveViewer.SLIDER_VEFF_MIN_MAX     = 8    # 10^8,  default: 8
# DPPUv2InteractiveViewer.SLIDER_VEFF_MIN_DEFAULT = 1e3  # default: 1e3

# DPPUv2InteractiveViewer.SLIDER_VEFF_MAX_MIN     = 2    # 10^2,  default: 2
# DPPUv2InteractiveViewer.SLIDER_VEFF_MAX_MAX     = 6    # 10^6,  default: 6
# DPPUv2InteractiveViewer.SLIDER_VEFF_MAX_DEFAULT = 2e3  # default: 2e3

# ── Geometric Parameters (NEW in v4) ──
# DPPUv2InteractiveViewer.SLIDER_EPSILON_MIN     = -0.9  # default: -0.9
# DPPUv2InteractiveViewer.SLIDER_EPSILON_MAX     = 3.0   # default: 3.0
# DPPUv2InteractiveViewer.SLIDER_EPSILON_DEFAULT = 0.0   # default: 0.0

# DPPUv2InteractiveViewer.SLIDER_ALPHA_W_MIN     = -5.0  # default: -5.0
# DPPUv2InteractiveViewer.SLIDER_ALPHA_W_MAX     = 5.0   # default: 5.0
# DPPUv2InteractiveViewer.SLIDER_ALPHA_W_DEFAULT = 0.0   # default: 0.0

# =============================================================================
# Launch the viewer
# =============================================================================
viewer = DPPUv2InteractiveViewer()
viewer.display()